In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:

class ConvBlock(nn.Module):
    """
    Common Convolution Block
    """
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=not norm)]
        if norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.1, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class FeaturePyramidCNN(nn.Module):
    """
    Shared CNN backbone for all input images.
    Input:  [B, 3, H, W]
    Output: [B, C, H/8, W/8]
    """
    def __init__(self, in_ch=3, base_ch=16, out_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(in_ch, base_ch, k=3, s=2, p=1),          # H/2
            ConvBlock(base_ch, base_ch, k=3, s=1, p=1),

            ConvBlock(base_ch, base_ch * 2, k=3, s=2, p=1),    # H/4
            ConvBlock(base_ch * 2, base_ch * 2, k=3, s=1, p=1),

            ConvBlock(base_ch * 2, out_ch, k=3, s=2, p=1),     # H/8
            ConvBlock(out_ch, out_ch, k=3, s=1, p=1),
        )

    def forward(self, x):
        return self.net(x)


class LocalCorrelation(nn.Module):
    """
    PWC-style local correlation.
    For each pixel in feat1, correlate with a local window in feat2.

    Input:
        feat1, feat2: [B, C, H, W]
    Output:
        corr volume:  [B, (2r+1)^2, H, W]
    """
    def __init__(self, radius=4):
        super().__init__()
        self.radius = radius

    def forward(self, feat1, feat2):
        B, C, H, W = feat1.shape
        r = self.radius

        # Normalize across channel for more stable dot product
        feat1 = F.normalize(feat1, p=2, dim=1)
        feat2 = F.normalize(feat2, p=2, dim=1)

        feat2_pad = F.pad(feat2, (r, r, r, r))
        corr_list = []

        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                shifted = feat2_pad[:, :, dy + r:dy + r + H, dx + r:dx + r + W]
                corr = torch.sum(feat1 * shifted, dim=1, keepdim=True)  # [B,1,H,W]
                corr_list.append(corr)

        corr_volume = torch.cat(corr_list, dim=1)  # [B, (2r+1)^2, H, W]
        return corr_volume


class PairwiseFlowEncoder(nn.Module):
    """
    First-stage encoder:
      image pair -> shared CNN -> correlation -> fused motion embedding

    Returns:
      pair_feat: [B, embed_ch, H/8, W/8]
      flow_init: [B, 2, H/8, W/8]   (optional coarse flow)
      corr:      [B, (2r+1)^2, H/8, W/8]
    """
    def __init__(self, feat_ch=64, corr_radius=4, embed_ch=128, predict_flow=True):
        super().__init__()
        self.feature_net = FeaturePyramidCNN(in_ch=3, base_ch=16, out_ch=feat_ch)
        self.corr = LocalCorrelation(radius=corr_radius)

        corr_ch = (2 * corr_radius + 1) ** 2
        fuse_in = feat_ch * 2 + corr_ch

        self.fuse = nn.Sequential(
            ConvBlock(fuse_in, 128, k=3, s=1, p=1),
            ConvBlock(128, embed_ch, k=3, s=1, p=1),
        )

        self.predict_flow = predict_flow
        if predict_flow:
            self.flow_head = nn.Sequential(
                ConvBlock(embed_ch, 64, k=3, s=1, p=1),
                nn.Conv2d(64, 2, kernel_size=3, stride=1, padding=1)
            )

    def forward(self, img1, img2):
        """
        img1, img2: [B, 3, H, W]
        """
        feat1 = self.feature_net(img1)   # [B, C, H/8, W/8]
        feat2 = self.feature_net(img2)   # [B, C, H/8, W/8]

        corr = self.corr(feat1, feat2)   # [B, corr_ch, H/8, W/8]

        fused = torch.cat([feat1, corr, feat2], dim=1)
        pair_feat = self.fuse(fused)

        if self.predict_flow:
            flow_init = self.flow_head(pair_feat)
        else:
            flow_init = None

        return {
            "feat1": feat1,
            "feat2": feat2,
            "corr": corr,
            "pair_feat": pair_feat,
            "flow_init": flow_init,
        }


class SequencePairEncoder(nn.Module):
    """
    Apply pairwise encoder over a sequence of frames.

    Input:
        imgs: [B, T, 3, H, W]

    Output:
        pair_feats: [B, T-1, C, H/8, W/8]
        flow_inits: [B, T-1, 2, H/8, W/8]
        corrs:      [B, T-1, K, H/8, W/8]
    """
    def __init__(self, feat_ch=64, corr_radius=4, embed_ch=128, predict_flow=True):
        super().__init__()
        self.pair_encoder = PairwiseFlowEncoder(
            feat_ch=feat_ch,
            corr_radius=corr_radius,
            embed_ch=embed_ch,
            predict_flow=predict_flow
        )

    def forward(self, imgs):
        B, T, C, H, W = imgs.shape
        assert C == 3, f"Expected RGB input, got C={C}"

        pair_feats = []
        flow_inits = []
        corrs = []

        for t in range(T - 1):
            out = self.pair_encoder(imgs[:, t], imgs[:, t + 1])
            pair_feats.append(out["pair_feat"])
            corrs.append(out["corr"])
            if out["flow_init"] is not None:
                flow_inits.append(out["flow_init"])

        pair_feats = torch.stack(pair_feats, dim=1)   # [B, T-1, embed_ch, H/8, W/8]
        corrs = torch.stack(corrs, dim=1)             # [B, T-1, corr_ch, H/8, W/8]

        if len(flow_inits) > 0:
            flow_inits = torch.stack(flow_inits, dim=1)
        else:
            flow_inits = None

        return {
            "pair_feats": pair_feats,
            "flow_inits": flow_inits,
            "corrs": corrs,
        }